In [126]:
import pandas as pd
import numpy as np
import regex as re
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
orders = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\orders.csv")
order_items = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\order_items.csv")
products = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\products (1).csv")
customers = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\customers.csv")
suppliers = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\suppliers (1).csv")

1. Pinpoint Key Product Availability Gaps: 
Identify which of the 200 products are most frequently out of stock or are major contributors 
to Cancelled orders, especially for shipments to Port Harcourt and surrounding areas, despite the high order volume.

In [3]:
#Getting the required columns only 
orders_1 = orders[["OrderID", "OrderStatus", 'ShippingCity']]
products_1 = products[["ProductID", "ProductName", "ProductStatus"]]
order_items_1 = order_items[["OrderID", "ProductID"]]

# Merging the dataframes
task1_df = (orders_1.merge(order_items_1, how='left')).merge(products_1, how='left')

task1_df.info() # Checking for any null values
# Checking columns for any issues 
set(task1_df['OrderStatus'])
set(task1_df['ProductStatus'])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163750 entries, 0 to 163749
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   OrderID        163750 non-null  object
 1   OrderStatus    163750 non-null  object
 2   ShippingCity   163750 non-null  object
 3   ProductID      163750 non-null  object
 4   ProductName    163750 non-null  object
 5   ProductStatus  163750 non-null  object
dtypes: object(6)
memory usage: 7.5+ MB


{'Active', 'OutOfStock', 'active'}

In [4]:
# I created a new column for OrderStatus, i didn't want to disrupt the original contents: 
# I don't know what they stand for, I'll be using this new column
# And I fixed the ProductStatus column as well

def fix(x):
    if x in['canceled',
 'canceled (old)',
 'canceled_v2',
 'cancelled',
 'cancelled (old)',
 'cancelled_v2']:
            return 'Canceled'
            
    elif x in['delivered',
'delivered (old)',
'delivered_v2',]:
        return 'Delivered'
    
    elif x in['partially shipped',
'partially shipped (old)',
'partially shipped_v2']:
        return 'Partially Shipped'
    
    elif x in['pending',
'pending (old)',
'pending_v2',]:
        return "Pending"
    
    elif x in['processing',
'processing (old)',
'processing_v2',]:
        return 'Processing'

    elif x in['returned',
'returned (old)',
'returned_v2',]:
        return 'Returned'

    elif x in['shipped',
'shipped (old)',
'shipped_v2']:
        return "Shipped"

task1_df['OrderStatus_1'] = (task1_df['OrderStatus'].str.lower().str.strip().apply(fix)).str.lower()
task1_df['ProductStatus'] = task1_df['ProductStatus'].str.capitalize()
task1_df.drop('OrderStatus', axis=1, inplace=True)
set(task1_df['OrderStatus_1'])
set(task1_df['ProductStatus'])

{'Active', 'Outofstock'}

Identify which of the 200 products are most frequently out of stock or are major contributors 
to Cancelled orders, especially for shipments to Port Harcourt and surrounding areas, despite the high order volume.

In [5]:
# I'm trying to determine how much each product has canceled orders compared to total number of orders
# I noticed most of the products have similar ratios of canceled orders  
canceled = pd.pivot_table(task1_df.query("ShippingCity in ['Aba', 'Owerri', 'Uyo','Port Harcourt']"), 
                          index = ['ProductName'], values=['ProductStatus'], columns=['OrderStatus_1'], aggfunc='count')
canceled['Total'] = canceled.sum(axis=1)
# cancelation ratio

canceled['canc_ratio'] = canceled['ProductStatus']['canceled']/canceled['Total']

# How high from the mean? 
canceled['cont'] = canceled['canc_ratio'] - canceled['canc_ratio'].mean()

canceled.loc[canceled['cont']>0.03] #These products contribute the highest to canceled orders

out_of_stock = pd.pivot_table(task1_df.query("ShippingCity in ['Aba','Owerri','Uyo','Port Harcourt']& ProductStatus =='Outofstock'"), 
                              index = ['ProductName'], values=['ProductID'], columns=['ProductStatus'], aggfunc='count')
out_of_stock = out_of_stock.sort_values(('ProductID', 'Outofstock'), ascending=False)
out_of_stock_1 = out_of_stock.reset_index()
products_with_availability_issues = set(out_of_stock_1['ProductName'])
canceled_1 = canceled.reset_index() 
products_with_availability_issues.update(set(canceled_1.loc[canceled_1['cont']>0.05,'ProductName']))
products_with_availability_issues

{'Adire Tv From Total Pro',
 'All Democratic Write Power Deluxe',
 'Ankara Half Of Pro',
 'Anyone Body Last Pro',
 'Avoid Perform Hand 833',
 'Beaded Away Where Eco',
 'Beaded Letter Clearly Basic',
 'Beaded Network Result Deluxe',
 'Believe National Put Pro',
 'Carved Section Bank 422',
 'Catch Shoulder Least A Eco',
 'Clearly Move 585',
 'Cost Can Eco',
 'Difficult Speak Seek Plus',
 'Dinner Baby Him Pro',
 'Drive Agent One Assume Plus',
 'Final Sell Father Plus',
 'Floor Total Basic',
 'Foreign Scientist Set Value Basic',
 'History Everyone Very Rise Plus',
 'It Sense Instead Whatever Basic',
 'Local Effect 788',
 'Mr Idea Indicate Goal Plus',
 'Out Material Person Subject Plus',
 'Produce Enter Property Eco',
 'Push Job Man System Plus',
 'Raise Shoulder 899',
 'Report Drop Green Eco',
 'Second Necessary Unit Pro',
 'Several Consider Pro',
 'Support Four Father Available Eco',
 'Then Purpose Same Well Deluxe',
 'Tonight Price Eco',
 'Total Mouth Basic',
 'Traditional Develop Appear

0.01820039671742526
0.2247325574742872

2. Assess Impact on Recent Customer Cohorts: 
Determine if fulfillment issues (e.g., significant delays where ActualDeliveryDate far exceeds ExpectedDeliveryDate, 
or high cancellation rates) are disproportionately affecting customers acquired since March 2024 (RegistrationDate > 2024-03-01), 
and if this correlates with lower initial repeat purchase rates from these new customers.

In [6]:
# For this objective, I need only the orders dataset and the customers dataset
# I'm taking the columns I need and merging the 2 datasets
orders_2 = orders[['OrderID', 'CustomerID','OrderDate','ExpectedDeliveryDate', 'ActualDeliveryDate', 'OrderStatus']]
customers_1 = customers[['CustomerID', 'RegistrationDate']]
# Some of the CustomerIDs have _DUP 
customers_1['CustomerID'] = customers_1['CustomerID'].str.replace('_DUP', '', regex=False)
task2_df = orders_2.merge(customers_1, 'left')

C:\Users\ogund\AppData\Local\Temp\ipykernel_23960\3336782027.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customers_1['CustomerID'] = customers_1['CustomerID'].str.replace('_DUP', '', regex=False)


In [7]:
# I'm converting the date columns to datetime type
task2_df['ExpectedDeliveryDate'] = pd.to_datetime(task2_df['ExpectedDeliveryDate'], format='mixed', errors='coerce')
task2_df['ActualDeliveryDate'] = pd.to_datetime(task2_df['ActualDeliveryDate'], format='mixed', errors='coerce')
task2_df['RegistrationDate'] = pd.to_datetime(task2_df['RegistrationDate'], format='mixed', errors='coerce')
task2_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   OrderID               75000 non-null  object        
 1   CustomerID            75000 non-null  object        
 2   OrderDate             75000 non-null  object        
 3   ExpectedDeliveryDate  45696 non-null  datetime64[ns]
 4   ActualDeliveryDate    23941 non-null  datetime64[ns]
 5   OrderStatus           75000 non-null  object        
 6   RegistrationDate      75000 non-null  datetime64[ns]
dtypes: datetime64[ns](3), object(4)
memory usage: 4.0+ MB


In [8]:
date = '2024-03-01'
task2_df['cohort'] = np.where(
            task2_df['RegistrationDate'] >= date,
            'New',
            'Old')
task2_df['delay_days_class'] = np.where((task2_df['ActualDeliveryDate'] - task2_df['ExpectedDeliveryDate']).dt.days<2, 'Okay', 'Critical')
task2_df['delay_days'] = (task2_df['ActualDeliveryDate'] - task2_df['ExpectedDeliveryDate']).dt.days
# task2_df.drop('OrderID',axis=1, inplace=True)
task2_df.head()

,OrderID,CustomerID,OrderDate,ExpectedDeliveryDate,ActualDeliveryDate,OrderStatus,RegistrationDate,cohort,delay_days_class,delay_days
0,ORD00001,C00167,2024-03-01 00:05:00,2024-03-02,2024-03-02 12:05:00,Returned,2025-02-05,New,Okay,0.0
1,ORD00002,C11600,2024-03-01 00:43:00,2024-03-04,NaT,Processing,2023-12-09,Old,Critical,NaN
2,ORD00003,C00451,2024-03-01 00:47:00,NaT,NaT,Canceled,2024-01-20,Old,Critical,NaN
3,ORD00004,C00244,2024-03-01 01:12:00,2024-03-08,2024-03-08 03:12:00,Delivered,2023-11-13,Old,Okay,0.0
4,ORD00005,C00200,2024-03-01 01:50:00,NaT,NaT,Cancelled,2024-09-13,New,Critical,NaN


In [9]:
a = pd.pivot_table(task2_df, columns='delay_days_class', aggfunc='count', index='cohort', values='CustomerID')
a['ratio'] = a['Critical']/a.sum(axis=1)
a
#The new customers don't suffer from delay issues a lot more than the old customers

delay_days_class,Critical,Okay,ratio
cohort,,,
New,39847,11282,0.779342
Old,18541,5330,0.776717


In [10]:
task2_df['OrderStatus_1'] = (task2_df['OrderStatus'].str.lower().str.strip().apply(fix)).str.lower()
b = pd.pivot_table(task2_df, columns='OrderStatus_1', aggfunc='count', index='cohort', values='CustomerID')
b['ratio'] = b['canceled']/b.sum(axis=1)
b
#The new customers don't suffer from order cancelation more than the old customers

OrderStatus_1,canceled,delivered,partially shipped,pending,processing,returned,shipped,ratio
cohort,,,,,,,,
New,11401,11321,5669,5655,5621,5775,5687,0.222985
Old,5356,5374,2609,2567,2641,2681,2643,0.224373


In [11]:
# I want to havea column that has repeat purchase with values, true and false. I will be doing this by grouping by customerID, 
# then I'll have a column for orderDate that will be the count of orderdates. I'll then check if OrderDates count > 1 is in my dataset
task2_df['OrderDate'] = pd.to_datetime(task2_df['OrderDate'], format='mixed', errors='coerce')
order_counts = task2_df.groupby('CustomerID').aggregate({'OrderDate': 'count'})
order_counts = order_counts.reset_index()
repeat_customer_ID = order_counts.loc[order_counts['OrderDate'] > 1]
task2_df['repeat_customer'] = task2_df['CustomerID'].isin(repeat_customer_ID['CustomerID'])
correlation = task2_df['delay_days'].corr(task2_df['repeat_customer'])
print(f"Correlation between Delay and Retention: {correlation:.4f}")

Correlation between Delay and Retention: -0.0045


3. Identify Top Supplier-Related Fulfillment Constraints: 
For the limited set of 15 suppliers, determine which ones are linked to the products experiencing 
the most severe availability gaps or quality issues (inferred from ReturnStatus) that impede smooth order fulfillment to the Port Harcourt market.

**`Availability gaps or quality issues`**

Avalilability - Out of stock
Quality Issues - Return status

### **`I'm thinking of including the products whose delay days are more than the normal range Also adding products that are out of stock, products that contribute highest to cancelation, and products that are returned due to quality issues`**

In [171]:
task3_df = ((orders.merge(order_items)).merge(products)).merge(suppliers)
task3_df.keys()

Index(['OrderID', 'CustomerID', 'OrderDate', 'ShipDate',
       'ExpectedDeliveryDate', 'ActualDeliveryDate', 'OrderStatus',
       'ShippingMethod', 'ShippingCost_NGN', 'ShippingAddress', 'ShippingCity',
       'ShippingState', 'ShippingPostalCode', 'ShippingCountry',
       'PaymentMethod', 'PaymentStatus', 'DiscountCode', 'DiscountAmount_NGN',
       'TotalAmount_NGN', 'OrderItemID', 'ProductID', 'Quantity',
       'UnitPriceAtPurchase_NGN', 'TotalItemPrice_NGN', 'ReturnStatus',
       'ProductName', 'Category', 'SupplierID', 'UnitPrice_NGN',
       'StockQuantity', 'ProductStatus', 'LaunchDate', 'Weight_kg',
       'SupplierName', 'Country', 'Region', 'ContactEmail', 'Phone',
       'YearsInBusiness', 'SupplierRating'],
      dtype='object')

In [172]:
task3_df = task3_df[['OrderID','ExpectedDeliveryDate', 'ActualDeliveryDate', 'OrderStatus',
                     'ShippingCity','OrderItemID', 'ProductID','ProductName','Quantity','ReturnStatus', 'Category', 
                     'SupplierID', 'StockQuantity', 'ProductStatus','SupplierName']]

task3_df['OrderStatus_1'] = (task3_df['OrderStatus'].str.lower().str.strip().apply(fix)).str.lower()
task3_df['ProductStatus'] = task3_df['ProductStatus'].str.capitalize()
# task3_df.drop('OrderStatus', axis=1, inplace=True)
set(task3_df['OrderStatus_1'])
set(task3_df['ProductStatus'])

{'Active', 'Outofstock'}

In [173]:
# I'm converting the date columns to datetime type
task3_df['ExpectedDeliveryDate'] = pd.to_datetime(task3_df['ExpectedDeliveryDate'], format='mixed', errors='coerce')
task3_df['ActualDeliveryDate'] = pd.to_datetime(task3_df['ActualDeliveryDate'], format='mixed', errors='coerce')
task3_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163750 entries, 0 to 163749
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   OrderID               163750 non-null  object        
 1   ExpectedDeliveryDate  99528 non-null   datetime64[ns]
 2   ActualDeliveryDate    51997 non-null   datetime64[ns]
 3   OrderStatus           163750 non-null  object        
 4   ShippingCity          163750 non-null  object        
 5   OrderItemID           163750 non-null  object        
 6   ProductID             163750 non-null  object        
 7   ProductName           163750 non-null  object        
 8   Quantity              163750 non-null  int64         
 9   ReturnStatus          24645 non-null   object        
 10  Category              163750 non-null  object        
 11  SupplierID            163750 non-null  object        
 12  StockQuantity         145528 non-null  float64       
 13 

In [31]:
task3_df['delay_days'] = (task3_df['ActualDeliveryDate'] - task3_df['ExpectedDeliveryDate']).dt.days

,OrderID,ExpectedDeliveryDate,ActualDeliveryDate,OrderStatus,ShippingCity,OrderItemID,ProductID,ProductName,Quantity,ReturnStatus,Category,SupplierID,StockQuantity,ProductStatus,SupplierName,OrderStatus_1,delay_days


In [266]:
products_with_severe_return_issues = pd.pivot_table(task3_df, columns='ReturnStatus',index='ProductName', aggfunc='count', values='ProductID')
products_with_severe_return_issues['ratio'] = products_with_severe_return_issues[['Approved','Completed']].sum(axis=1, skipna=True)/products_with_severe_return_issues.sum(axis=1, skipna=True)
products_with_severe_return_issues = products_with_severe_return_issues.sort_values(by='ratio', ascending=False).head(10)
a = set(products_with_severe_return_issues.index)
len(a)

10

In [267]:
task3_df['delay'] = np.where((task3_df['ActualDeliveryDate']-task3_df['ExpectedDeliveryDate']).dt.days > 1, 'Critical','Okay')
task3_df.head()
products_with_severe_delivery_issues= pd.pivot_table(task3_df, columns='delay', index='ProductName', values='ProductID', aggfunc='count')
products_with_severe_delivery_issues['ratio'] = products_with_severe_delivery_issues['Critical']/products_with_severe_delivery_issues.sum(axis=1, skipna=True)
products_with_severe_delivery_issues = products_with_severe_delivery_issues.sort_values(by='ratio', ascending=False).head(10)
b = set(products_with_severe_delivery_issues.index)
len(b)

10

In [268]:
# task3_df['OrderStatus_1'] = (task3_df['OrderStatus'].str.lower().str.strip().apply(fix)).str.lower()
# task3_df['ProductStatus'] = task3_df['ProductStatus'].str.capitalize()
# task3_df.drop('OrderStatus', axis=1, inplace=True)
products_with_severe_cancelation_issues = pd.pivot_table(task3_df, columns='OrderStatus_1', values='ProductID', index='ProductName', aggfunc='count')
products_with_severe_cancelation_issues['ratio'] = products_with_severe_cancelation_issues['canceled']/products_with_severe_cancelation_issues[[
    'canceled', 'delivered', 'returned']].sum(axis=1)
products_with_severe_cancelation_issues = products_with_severe_cancelation_issues.sort_values(by='ratio', ascending=False).head(10)
c = set(products_with_severe_cancelation_issues.index)
len(c)

10

In [269]:
products_with_severe_availability_issues = pd.pivot_table(task3_df, index='ProductName', columns='OrderStatus_1', values='StockQuantity', aggfunc='sum').sum(axis=1)
d = set(products_with_severe_availability_issues[products_with_severe_availability_issues <= 0].index)
len(d)

16

In [275]:
products_with_severe_issues = a.union(b,c,d)
#products_with_severe_issues = PWSI
suppliers_linked_with_PWSI = set(task3_df.query("ProductName.isin(@products) & ShippingCity == 'Port Harcourt'")['SupplierName'])
suppliers_linked_with_PWSI

{'Bowers and Sons',
 'Brown Group',
 'Bush Ltd',
 'Byrd-Perkins',
 'Fitzpatrick Ltd',
 'Harrison Inc',
 'Lopez-Jones',
 'Mata-Hill',
 'Nguyen-Matthews',
 'Parks, Gomez and Lowery',
 'Reyes-Ochoa'}

In [235]:
# I wonder if the return in OrderStatus has anything to do with the returnstatus column
task3_df.loc[task3_df['OrderStatus_1']=='returned', 'ReturnStatus'].dropna()
# I would run a corr on this later
# task3_df['OrderStatus_1'].corr(task3_df['ReturnStatus'])

24        Not Returned
145       Not Returned
264       Not Returned
269       Not Returned
323       Not Returned
              ...     
163189    Not Returned
163345       Completed
163459    Not Returned
163580    Not Returned
163594    Not Returned
Name: ReturnStatus, Length: 2719, dtype: object

In [ ]:
import pandas as pd
import numpy as np
import regex as re
import matplotlib.pyplot as plt
%matplotlib inline

orders = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\orders.csv")
order_items = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\order_items.csv")
products = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\products (1).csv")
customers = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\customers.csv")
suppliers = pd.read_csv(r"C:\Users\ogund\OneDrive\Desktop\SUPPLY\suppliers (1).csv")

#Getting the required columns only 
orders_1 = orders[["OrderID", "OrderStatus", 'ShippingCity']]
products_1 = products[["ProductID", "ProductName", "ProductStatus"]]
order_items_1 = order_items[["OrderID", "ProductID"]]

# Merging the dataframes
task1_df = (orders_1.merge(order_items_1, how='left')).merge(products_1, how='left')

task1_df.info() # Checking for any null values
# Checking columns for any issues 
set(task1_df['OrderStatus'])
set(task1_df['ProductStatus'])

# I created a new column for OrderStatus, i didn't want to disrupt the original contents: 
# I don't know what they stand for, I'll be using this new column
# And I fixed the ProductStatus column as well

def fix(x):
    if x in['canceled',
 'canceled (old)',
 'canceled_v2',
 'cancelled',
 'cancelled (old)',
 'cancelled_v2']:
            return 'Canceled'
            
    elif x in['delivered',
'delivered (old)',
'delivered_v2',]:
        return 'Delivered'
    
    elif x in['partially shipped',
'partially shipped (old)',
'partially shipped_v2']:
        return 'Partially Shipped'
    
    elif x in['pending',
'pending (old)',
'pending_v2',]:
        return "Pending"
    
    elif x in['processing',
'processing (old)',
'processing_v2',]:
        return 'Processing'

    elif x in['returned',
'returned (old)',
'returned_v2',]:
        return 'Returned'

    elif x in['shipped',
'shipped (old)',
'shipped_v2']:
        return "Shipped"

task1_df['OrderStatus_1'] = (task1_df['OrderStatus'].str.lower().str.strip().apply(fix)).str.lower()
task1_df['ProductStatus'] = task1_df['ProductStatus'].str.capitalize()
task1_df.drop('OrderStatus', axis=1, inplace=True)
set(task1_df['OrderStatus_1'])
set(task1_df['ProductStatus'])

# I'm trying to determine how much each product has canceled orders compared to total number of orders
# I noticed most of the products have similar ratios of canceled orders  
canceled = pd.pivot_table(task1_df.query("ShippingCity in ['Aba', 'Owerri', 'Uyo','Port Harcourt']"), 
                          index = ['ProductName'], values=['ProductStatus'], columns=['OrderStatus_1'], aggfunc='count')
canceled['Total'] = canceled.sum(axis=1)
# cancelation ratio

canceled['canc_ratio'] = canceled['ProductStatus']['canceled']/canceled['Total']

# How high from the mean? 
canceled['cont'] = canceled['canc_ratio'] - canceled['canc_ratio'].mean()

canceled.loc[canceled['cont']>0.03] #These products contribute the highest to canceled orders

out_of_stock = pd.pivot_table(task1_df.query("ShippingCity in ['Aba','Owerri','Uyo','Port Harcourt']& ProductStatus =='Outofstock'"), 
                              index = ['ProductName'], values=['ProductID'], columns=['ProductStatus'], aggfunc='count')
out_of_stock = out_of_stock.sort_values(('ProductID', 'Outofstock'), ascending=False)
out_of_stock_1 = out_of_stock.reset_index()
products_with_availability_issues = set(out_of_stock_1['ProductName'])
canceled_1 = canceled.reset_index() 
products_with_availability_issues.update(set(canceled_1.loc[canceled_1['cont']>0.05,'ProductName']))
products_with_availability_issues


# For this objective, I need only the orders dataset and the customers dataset
# I'm taking the columns I need and merging the 2 datasets
orders_2 = orders[['OrderID', 'CustomerID','OrderDate','ExpectedDeliveryDate', 'ActualDeliveryDate', 'OrderStatus']]
customers_1 = customers[['CustomerID', 'RegistrationDate']]
# Some of the CustomerIDs have _DUP 
customers_1['CustomerID'] = customers_1['CustomerID'].str.replace('_DUP', '', regex=False)
task2_df = orders_2.merge(customers_1, 'left')

# I'm converting the date columns to datetime type
task2_df['ExpectedDeliveryDate'] = pd.to_datetime(task2_df['ExpectedDeliveryDate'], format='mixed', errors='coerce')
task2_df['ActualDeliveryDate'] = pd.to_datetime(task2_df['ActualDeliveryDate'], format='mixed', errors='coerce')
task2_df['RegistrationDate'] = pd.to_datetime(task2_df['RegistrationDate'], format='mixed', errors='coerce')
task2_df.info()

date = '2024-03-01'
task2_df['cohort'] = np.where(
            task2_df['RegistrationDate'] >= date,
            'New',
            'Old')
task2_df['delay_days_class'] = np.where((task2_df['ActualDeliveryDate'] - task2_df['ExpectedDeliveryDate']).dt.days<2, 'Okay', 'Critical')
task2_df['delay_days'] = (task2_df['ActualDeliveryDate'] - task2_df['ExpectedDeliveryDate']).dt.days
# task2_df.drop('OrderID',axis=1, inplace=True)
task2_df.head()

a = pd.pivot_table(task2_df, columns='delay_days_class', aggfunc='count', index='cohort', values='CustomerID')
a['ratio'] = a['Critical']/a.sum(axis=1)
a
#The new customers don't suffer from delay issues a lot more than the old customers

task2_df['OrderStatus_1'] = (task2_df['OrderStatus'].str.lower().str.strip().apply(fix)).str.lower()
b = pd.pivot_table(task2_df, columns='OrderStatus_1', aggfunc='count', index='cohort', values='CustomerID')
b['ratio'] = b['canceled']/b.sum(axis=1)
b
#The new customers don't suffer from order cancelation more than the old customers

# I want to havea column that has repeat purchase with values, true and false. I will be doing this by grouping by customerID, 
# then I'll have a column for orderDate that will be the count of orderdates. I'll then check if OrderDates count > 1 is in my dataset
task2_df['OrderDate'] = pd.to_datetime(task2_df['OrderDate'], format='mixed', errors='coerce')
order_counts = task2_df.groupby('CustomerID').aggregate({'OrderDate': 'count'})
order_counts = order_counts.reset_index()
repeat_customer_ID = order_counts.loc[order_counts['OrderDate'] > 1]
task2_df['repeat_customer'] = task2_df['CustomerID'].isin(repeat_customer_ID['CustomerID'])
correlation = task2_df['delay_days'].corr(task2_df['repeat_customer'])
print(f"Correlation between Delay and Retention: {correlation:.4f}")

task3_df = ((orders.merge(order_items)).merge(products)).merge(suppliers)
task3_df.keys()

task3_df = task3_df[['OrderID','ExpectedDeliveryDate', 'ActualDeliveryDate', 'OrderStatus',
                     'ShippingCity','OrderItemID', 'ProductID','ProductName','Quantity','ReturnStatus', 'Category', 
                     'SupplierID', 'StockQuantity', 'ProductStatus','SupplierName']]

task3_df['OrderStatus_1'] = (task3_df['OrderStatus'].str.lower().str.strip().apply(fix)).str.lower()
task3_df['ProductStatus'] = task3_df['ProductStatus'].str.capitalize()
# task3_df.drop('OrderStatus', axis=1, inplace=True)
set(task3_df['OrderStatus_1'])
set(task3_df['ProductStatus'])

# I'm converting the date columns to datetime type
task3_df['ExpectedDeliveryDate'] = pd.to_datetime(task3_df['ExpectedDeliveryDate'], format='mixed', errors='coerce')
task3_df['ActualDeliveryDate'] = pd.to_datetime(task3_df['ActualDeliveryDate'], format='mixed', errors='coerce')
task3_df.info()

products_with_severe_return_issues = pd.pivot_table(task3_df, columns='ReturnStatus',index='ProductName', aggfunc='count', values='ProductID')
products_with_severe_return_issues['ratio'] = products_with_severe_return_issues[['Approved','Completed']].sum(axis=1, skipna=True)/products_with_severe_return_issues.sum(axis=1, skipna=True)
products_with_severe_return_issues = products_with_severe_return_issues.sort_values(by='ratio', ascending=False).head(10)
a = set(products_with_severe_return_issues.index)
len(a)

task3_df['delay'] = np.where((task3_df['ActualDeliveryDate']-task3_df['ExpectedDeliveryDate']).dt.days > 1, 'Critical','Okay')
task3_df.head()
products_with_severe_delivery_issues= pd.pivot_table(task3_df, columns='delay', index='ProductName', values='ProductID', aggfunc='count')
products_with_severe_delivery_issues['ratio'] = products_with_severe_delivery_issues['Critical']/products_with_severe_delivery_issues.sum(axis=1, skipna=True)
products_with_severe_delivery_issues = products_with_severe_delivery_issues.sort_values(by='ratio', ascending=False).head(10)
b = set(products_with_severe_delivery_issues.index)
len(b)

# task3_df['OrderStatus_1'] = (task3_df['OrderStatus'].str.lower().str.strip().apply(fix)).str.lower()
# task3_df['ProductStatus'] = task3_df['ProductStatus'].str.capitalize()
# task3_df.drop('OrderStatus', axis=1, inplace=True)
products_with_severe_cancelation_issues = pd.pivot_table(task3_df, columns='OrderStatus_1', values='ProductID', index='ProductName', aggfunc='count')
products_with_severe_cancelation_issues['ratio'] = products_with_severe_cancelation_issues['canceled']/products_with_severe_cancelation_issues[[
    'canceled', 'delivered', 'returned']].sum(axis=1)
products_with_severe_cancelation_issues = products_with_severe_cancelation_issues.sort_values(by='ratio', ascending=False).head(10)
c = set(products_with_severe_cancelation_issues.index)
len(c)

products_with_severe_availability_issues = pd.pivot_table(task3_df, index='ProductName', columns='OrderStatus_1', values='StockQuantity', aggfunc='sum').sum(axis=1)
d = set(products_with_severe_availability_issues[products_with_severe_availability_issues <= 0].index)
len(d)

products_with_severe_issues = a.union(b,c,d)
#products_with_severe_issues = PWSI
suppliers_linked_with_PWSI = set(task3_df.query("ProductName.isin(@products) & ShippingCity == 'Port Harcourt'")['SupplierName'])
suppliers_linked_with_PWSI